# Generate Your Own Steering Dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/modulabs-personalab/psyctl/blob/main/examples/en/03_generate_dataset.ipynb)

Steering vectors are trained on **contrastive pairs**: for the same situation, a "positive" response exhibits the target personality and a "neutral" response is the baseline. PSYCTL generates these pairs from the [allenai/soda](https://huggingface.co/datasets/allenai/soda) dialogue dataset.

**Two paths to generate data:**
- **Path A: OpenRouter API** — No GPU required, faster with parallel workers, costs ~$0.01-0.05 per 50 samples
- **Path B: Local Model** — Free, requires GPU, slower

**In this notebook you will:**
1. Understand the contrastive dataset structure
2. Generate a dataset using OpenRouter API or a local model
3. Inspect the generated data
4. (Optional) Upload to HuggingFace Hub

**Time:** ~5 minutes

In [ ]:
!pip install -q git+https://github.com/modulabs-personalab/psyctl.git bitsandbytes accelerate

In [ ]:
import os

try:
    from google.colab import userdata
    _hf = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _hf if isinstance(_hf, str) else str(_hf)
    print("Loaded HF_TOKEN from Colab Secrets")
except Exception:
    if not os.environ.get("HF_TOKEN"):
        try:
            import getpass
            os.environ["HF_TOKEN"] = getpass.getpass("Enter your HuggingFace token: ")
        except Exception:
            raise RuntimeError(
                "HF_TOKEN not found. Add it to Colab Secrets (key icon in left sidebar) "
                "and enable notebook access, then re-run this cell."
            )

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

# Optional: OpenRouter API key (for Path A)
try:
    from google.colab import userdata
    _or = userdata.get("OPENROUTER_API_KEY")
    os.environ["OPENROUTER_API_KEY"] = _or if isinstance(_or, str) else str(_or)
    print("Loaded OPENROUTER_API_KEY from Colab Secrets")
except Exception:
    pass

if not os.environ.get("OPENROUTER_API_KEY"):
    print("OPENROUTER_API_KEY not set. You can still use Path B (local model).")
    print("To set it: Add OPENROUTER_API_KEY to Colab Secrets, or enter it below.")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PSYCTL_LOG_LEVEL"] = "WARNING"

## Dataset Structure

Each sample in a steering dataset is a JSONL record with these fields:

```json
{
  "situation": "Alice and Bob are at a coffee shop. Alice says: 'I can't believe...'",
  "char_name": "Bob",
  "positive": "Response exhibiting the target personality trait",
  "neutral": "Neutral baseline response"
}
```

The **positive** response is generated by prompting the model to roleplay as a character with a specific personality (e.g., "extremely agreeable"), while the **neutral** response uses a plain character description. The difference between these activations is what becomes the steering vector.

## Path A: OpenRouter API (No GPU Required)

Uses a cloud model via [OpenRouter](https://openrouter.ai/). Fast with parallel workers. Requires an API key.

In [ ]:
from pathlib import Path
from psyctl.core.dataset_builder import DatasetBuilder

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    import getpass
    OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key (or press Enter to skip): ")
    if OPENROUTER_API_KEY:
        os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

if OPENROUTER_API_KEY:
    OUTPUT_DIR = Path("./dataset_openrouter")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    builder = DatasetBuilder(
        use_openrouter=True,
        openrouter_api_key=OPENROUTER_API_KEY,
        openrouter_max_workers=4,
    )

    dataset_file = builder.build_caa_dataset(
        model="qwen/qwen3-next-80b-a3b-instruct",
        personality="Extroversion",
        output_dir=OUTPUT_DIR,
        limit_samples=10,
        dataset_name="allenai/soda",
        temperature=0.7,
        max_tokens=100,
    )
    print(f"Dataset generated: {dataset_file}")
else:
    print("Skipping Path A (no API key). Try Path B below.")

## Path B: Local Model (GPU Required)

Uses a small local model. Free but slower, requires a GPU runtime.

In [ ]:
OUTPUT_DIR_LOCAL = Path("./dataset_local")
OUTPUT_DIR_LOCAL.mkdir(parents=True, exist_ok=True)

builder_local = DatasetBuilder(use_openrouter=False)

dataset_file_local = builder_local.build_caa_dataset(
    model="google/gemma-3-270m-it",
    personality="Extroversion",
    output_dir=OUTPUT_DIR_LOCAL,
    limit_samples=10,
    dataset_name="allenai/soda",
    temperature=0.7,
    max_tokens=100,
)
print(f"Dataset generated: {dataset_file_local}")

## Inspect the Dataset

In [ ]:
import json
from pathlib import Path

# Use whichever dataset was generated above
dataset_path = None
for candidate in [Path("./dataset_openrouter"), Path("./dataset_local")]:
    if candidate.exists():
        jsonl_files = list(candidate.glob("*.jsonl"))
        if jsonl_files:
            dataset_path = jsonl_files[0]
            break

if dataset_path is None:
    print("No dataset found. Run Path A or Path B above first.")
else:
    samples = []
    with open(dataset_path, encoding="utf-8") as f:
        for line in f:
            samples.append(json.loads(line))

    print(f"File: {dataset_path}")
    print(f"Total samples: {len(samples)}")
    print(f"File size: {dataset_path.stat().st_size / 1024:.1f} KB")

    print("\n--- Sample 1 ---")
    s = samples[0]
    print(f"Character: {s['char_name']}")
    print(f"Situation: {s['situation'][:200]}...")
    print(f"Positive:  {s['positive']}")
    print(f"Neutral:   {s['neutral']}")

    if len(samples) > 1:
        print("\n--- Sample 2 ---")
        s = samples[1]
        print(f"Character: {s['char_name']}")
        print(f"Positive:  {s['positive']}")
        print(f"Neutral:   {s['neutral']}")

## (Optional) Upload to HuggingFace Hub

Share your dataset with the community. Requires a HuggingFace token with **write** permission.

In [ ]:
# Uncomment and modify to upload your dataset
#
# from psyctl.core.dataset_builder import DatasetBuilder
#
# builder = DatasetBuilder()
# repo_url = builder.upload_to_hub(
#     jsonl_file=dataset_path,
#     repo_id="your-username/extroversion-steering",  # Change this!
#     private=False,
#     license="mit",
# )
# print(f"Uploaded to: {repo_url}")